In [13]:
import sys, os, json, re, time
sys.path.append(r'C:\Users\umita\Desktop\PYT')
from umit_pylib import DB
from umit_pylib import df_lib as ME
from datetime import datetime as dtt
import pandas as pd, numpy as np


In [15]:
db = DB.create('SECONDARY', 'MULTI', 'v_name') 

SECONDARY MULTI SECONDARY_MULTI_0
PYLIB :: in_server SECONDARY name= SECONDARY_MULTI_0


In [173]:
v_itr1 = '2024-08-13'    
v_menkul = 'GARAN.E'
sql = """
    WITH tavtab_w AS (
    SELECT g.date trh, g.securitycode sec, MAX(tavan) tavan, MAX(taban) taban
      FROM (
        SELECT date, securitycode, field_ul::numeric tavan, NULL taban
          FROM bistulfield a
         WHERE date = '{v_itr1}' AND time < '10:00' AND securitycode = '{v_menkul}'    AND field_ul <> ''
        UNION ALL
        SELECT date, securitycode, NULL, field_ll::numeric
          FROM bistllfield a	  
         WHERE date = '{v_itr1}'  AND time < '10:00' AND securitycode = '{v_menkul}'    AND field_ll <> ''
        ) g
     GROUP BY g.date, g.securitycode
    )
    , orders_w AS (    
     SELECT securitycode, date, time, isbid, volume, tavan, taban, price, sira 
       FROM (
         SELECT securitycode, date, time, isbid, volume, tavan, taban,
             CASE WHEN isbid = 't' AND price = 0 THEN tavan + 1  
                  WHEN isbid = 'f' AND price = 0 THEN taban - 1 ELSE price END price, 
             RANK() OVER (PARTITION BY date, securitycode, orderid ORDER BY globaltransid DESC) sira 
           FROM bistorderlog a    
           LEFT JOIN tavtab_w b ON a.securitycode = b.sec AND a.date = b.trh
          WHERE date = '{v_itr1}' AND securitycode = '{v_menkul}' 
            AND amfof = false 
        ) a
      WHERE sira = 1
        AND volume > 0
     )    

    , book_w AS (
    SELECT date, securitycode, price, tavan, taban,
        SUM(CASE WHEN     b.isbid AND volume > 0 THEN 1 END) al_bek_say,
        SUM(CASE WHEN NOT b.isbid AND volume > 0 THEN 1 END) sat_bek_say,
        SUM(CASE WHEN     b.isbid THEN volume ELSE 0 END) al_bek_mkt,
        SUM(CASE WHEN NOT b.isbid THEN volume ELSE 0 END) sat_bek_mkt
      FROM orders_w b            
     GROUP BY date, securitycode, price, tavan, taban
    HAVING SUM(volume) > 0 
     ORDER BY date, securitycode, price
    )

    SELECT CURRENT_TIME+'03:00:00'::time sorgu, 
        taban, eniyi_alis, al_sira, price, sat_sira, eniyi_satis, tavan, 
        al_bek_say, al_bek_mkt, al_kum_mkt,   sat_bek_say, sat_bek_mkt, sat_kum_mkt 

      FROM (
        SELECT date, securitycode, price, tavan, taban, al_bek_say, sat_bek_say, al_bek_mkt, sat_bek_mkt, 
            SUM(al_bek_mkt) OVER (PARTITION BY date, securitycode ORDER BY price DESC) al_kum_mkt,
            SUM(sat_bek_mkt) OVER (PARTITION BY date, securitycode ORDER BY price ASC) sat_kum_mkt,
            SUM(CASE WHEN al_bek_mkt > 0 THEN 1 END) OVER (PARTITION BY date, securitycode ORDER BY price DESC) al_sira,
            SUM(CASE WHEN sat_bek_mkt > 0 THEN 1 END) OVER (PARTITION BY date, securitycode ORDER BY price ASC) sat_sira,
            MAX(CASE WHEN al_bek_mkt > 0 THEN price END) OVER (PARTITION BY date, securitycode) eniyi_alis,
            MIN(CASE WHEN sat_bek_mkt > 0 THEN price END) OVER (PARTITION BY date, securitycode) eniyi_satis
          FROM book_w
        ) a
     ORDER BY price

"""

df = db.get_df(sql,  {'v_itr1': v_itr1, 'v_menkul': v_menkul} )
df = df.fillna('')   
df_raw = df

In [124]:
df = df.set_index([df.index, 'sorgu'])

In [125]:
table_styler = [
    {   "selector" : "table",
        "props":[   ("border", "3px solid black"),
                    ("border-collapse","separate"),
                    ("width", "100%")
                ]
     },
     {   "selector" : "th",
         "props":[   ("color","black"),
                     ("background-color", "#F5F3F3"),
                     ("border", "1px solid gray"),
                     ("padding", "3px 3px"),
                     ("border-collapse","separate"),
                     ("font-size", "16px")
                 ]
     },
     {   "selector" : "td",
         "props":[   ("color","black"),
                     ("border", "1px solid gray"),
                     ("padding", "1px 3px"),
                     ("font-size", "14px")
                 ]
     }
]
df_styler = df.style.set_table_styles(table_styler)  \
    .format_index(str.title, axis=1) 


fyt_cols = 'taban eniyi_alis price eniyi_satis tavan'.split()
fmt = {x: "{:.2f}" for x in fyt_cols}
df_styler = df_styler.format(formatter=fmt, precision=0, thousands=".", decimal=",")  



df_styler

,,Taban,Eniyi_Alis,Al_Sira,Price,Sat_Sira,Eniyi_Satis,Tavan,Al_Bek_Say,Al_Bek_Mkt,Al_Kum_Mkt,Sat_Bek_Say,Sat_Bek_Mkt,Sat_Kum_Mkt
,sorgu,,,,,,,,,,,,,
0,10:26:51.373512+00:00,"101,60","116,20",112,"101,60",,"116,30","124,00",12,5.487,1.218.276,,0,0
1,10:26:51.373512+00:00,"101,60","116,20",111,"101,70",,"116,30","124,00",2,77,1.212.789,,0,0
2,10:26:51.373512+00:00,"101,60","116,20",110,"101,80",,"116,30","124,00",2,87,1.212.712,,0,0
3,10:26:51.373512+00:00,"101,60","116,20",109,"101,90",,"116,30","124,00",2,86,1.212.625,,0,0
4,10:26:51.373512+00:00,"101,60","116,20",108,"102,00",,"116,30","124,00",19,189.830,1.212.539,,0,0
5,10:26:51.373512+00:00,"101,60","116,20",107,"102,10",,"116,30","124,00",6,3.846,1.022.709,,0,0
6,10:26:51.373512+00:00,"101,60","116,20",106,"102,30",,"116,30","124,00",12,31,1.018.863,,0,0
7,10:26:51.373512+00:00,"101,60","116,20",105,"102,50",,"116,30","124,00",2,2.600,1.018.832,,0,0
8,10:26:51.373512+00:00,"101,60","116,20",104,"102,80",,"116,30","124,00",2,625,1.016.232,,0,0


In [126]:
import seaborn as sns
cm = sns.light_palette("green", as_cmap=True)

df_styler.background_gradient(subset=fyt_cols, cmap=cm) \
    .bar(subset=['sat_bek_mkt', 'sat_kum_mkt'], color='#d65f5f') \
    .bar(subset=['al_bek_mkt', 'al_kum_mkt'], color='teal') 

,,Taban,Eniyi_Alis,Al_Sira,Price,Sat_Sira,Eniyi_Satis,Tavan,Al_Bek_Say,Al_Bek_Mkt,Al_Kum_Mkt,Sat_Bek_Say,Sat_Bek_Mkt,Sat_Kum_Mkt
,sorgu,,,,,,,,,,,,,
0,10:26:51.373512+00:00,"101,60","116,20",112,"101,60",,"116,30","124,00",12,5.487,1.218.276,,0,0
1,10:26:51.373512+00:00,"101,60","116,20",111,"101,70",,"116,30","124,00",2,77,1.212.789,,0,0
2,10:26:51.373512+00:00,"101,60","116,20",110,"101,80",,"116,30","124,00",2,87,1.212.712,,0,0
3,10:26:51.373512+00:00,"101,60","116,20",109,"101,90",,"116,30","124,00",2,86,1.212.625,,0,0
4,10:26:51.373512+00:00,"101,60","116,20",108,"102,00",,"116,30","124,00",19,189.830,1.212.539,,0,0
5,10:26:51.373512+00:00,"101,60","116,20",107,"102,10",,"116,30","124,00",6,3.846,1.022.709,,0,0
6,10:26:51.373512+00:00,"101,60","116,20",106,"102,30",,"116,30","124,00",12,31,1.018.863,,0,0
7,10:26:51.373512+00:00,"101,60","116,20",105,"102,50",,"116,30","124,00",2,2.600,1.018.832,,0,0
8,10:26:51.373512+00:00,"101,60","116,20",104,"102,80",,"116,30","124,00",2,625,1.016.232,,0,0


In [103]:

df_styler.background_gradient(subset=fyt_cols, cmap=cm) \
    .bar(subset=['sat_bek_mkt'], align=0, vmin=-20000, vmax=20000, cmap="bwr", height=50,
              width=60, props="width: 120px; border-right: 1px solid black;")\
         .text_gradient(cmap="bwr", vmin=-200000, vmax=200000)

In [105]:
style1 = df.style\
            .applymap(style_negative, props='color:red;')\
            .map(lambda v: 'opacity: 20%;' if (v < 0.3) and (v > -0.3) else None)\
            .set_table_styles([{"selector": "th", "props": "color: blue;"}])\
            .hide(axis="index")
style1

NameError: name 'style_negative' is not defined

In [106]:
from ipywidgets import widgets
@widgets.interact
def f(h_neg=(0, 359, 1), h_pos=(0, 359), s=(0., 99.9), l=(0., 99.9)):
    return df.style.background_gradient(
        cmap=sns.palettes.diverging_palette(h_neg=h_neg, h_pos=h_pos, s=s, l=l,
                                            as_cmap=True)
    )

interactive(children=(IntSlider(value=179, description='h_neg', max=359), IntSlider(value=179, description='h_…

In [127]:

df_styler.set_sticky(axis="index", levels=[1,2,3])

,,Taban,Eniyi_Alis,Al_Sira,Price,Sat_Sira,Eniyi_Satis,Tavan,Al_Bek_Say,Al_Bek_Mkt,Al_Kum_Mkt,Sat_Bek_Say,Sat_Bek_Mkt,Sat_Kum_Mkt
,sorgu,,,,,,,,,,,,,
0,10:26:51.373512+00:00,"101,60","116,20",112,"101,60",,"116,30","124,00",12,5.487,1.218.276,,0,0
1,10:26:51.373512+00:00,"101,60","116,20",111,"101,70",,"116,30","124,00",2,77,1.212.789,,0,0
2,10:26:51.373512+00:00,"101,60","116,20",110,"101,80",,"116,30","124,00",2,87,1.212.712,,0,0
3,10:26:51.373512+00:00,"101,60","116,20",109,"101,90",,"116,30","124,00",2,86,1.212.625,,0,0
4,10:26:51.373512+00:00,"101,60","116,20",108,"102,00",,"116,30","124,00",19,189.830,1.212.539,,0,0
5,10:26:51.373512+00:00,"101,60","116,20",107,"102,10",,"116,30","124,00",6,3.846,1.022.709,,0,0
6,10:26:51.373512+00:00,"101,60","116,20",106,"102,30",,"116,30","124,00",12,31,1.018.863,,0,0
7,10:26:51.373512+00:00,"101,60","116,20",105,"102,50",,"116,30","124,00",2,2.600,1.018.832,,0,0
8,10:26:51.373512+00:00,"101,60","116,20",104,"102,80",,"116,30","124,00",2,625,1.016.232,,0,0


In [131]:
import pandas as pd
from datetime import datetime, timedelta


report_start_time = datetime.strptime('2022-10-18 15:00:00' , "%Y-%m-%d %H:%M:%S")  #datetime.now()


def highlight_cells(val, check_time, limit):
    time_diff = check_time - val

    total_hours = (time_diff.days*24) + (time_diff.seconds / (60*60))

    if (total_hours >= limit):
        format_code = '''background-color: #B00202;
           font-weight: bold'''
    else:
        format_code = ''

    return format_code

def highlight_snap(row, check_time):
    # subset=['SLA Domain Name', 'Last Snapshot']
    if row['Policy'] == 'NONPROD_BACKUP':
        style = highlight_cells(row['Snapshot Time'], check_time, 20)
    else:
        style = highlight_cells(row['Snapshot Time'], check_time, 24)

    return pd.Series({'Snapshot Time':style})

d = {'DB': ['A-DB', 'B-DB', 'C-DB'], 'Policy': ['PROD_BACKUP','PROD_BACKUP','NONPROD_BACKUP'], 'Snapshot Time': ['10/18/2022 12:00:00','10/16/2022 10:00:00','10/15/2022 16:00:00']}
df = pd.DataFrame(data=d)
df['Snapshot Time'] = pd.to_datetime(df['Snapshot Time'])

table_styler = [
    {   "selector" : "table",
        "props":[   ("border", "3px solid black"),
                    ("border-collapse","separate"),
                    ("width", "100%")
                ]
     },
     {   "selector" : "th",
         "props":[   ("color","black"),
                     ("background-color", "#F5F3F3"),
                     ("border", "1px solid gray"),
                     ("padding", "3px 3px"),
                     ("border-collapse","separate"),
                     ("font-size", "16px")
                 ]
     },
     {   "selector" : "td",
         "props":[   ("color","black"),
                     ("border", "1px solid gray"),
                     ("padding", "1px 3px"),
                     ("font-size", "14px")
                 ]
     }
]

df_styler = df.style.set_table_styles(table_styler)

df_styler = df_styler.apply(highlight_snap, subset=['Policy','Snapshot Time'], axis=1, check_time=report_start_time)

df_styler          #.to_html()

,DB,Policy,Snapshot Time
0,A-DB,PROD_BACKUP,2022-10-18 12:00:00
1,B-DB,PROD_BACKUP,2022-10-16 10:00:00
2,C-DB,NONPROD_BACKUP,2022-10-15 16:00:00


In [171]:
df.columns

table_styler = [
    {   "selector" : "table",
        "props":[   ("border", "3px solid black"),
                    ("border-collapse","separate"),
                    ("width", "100%")
                ]
     },
     {   "selector" : "th",
         "props":[   ("color","black"),
                     ("background-color", "#F5F3F3"),
                     ("border", "1px solid gray"),
                     ("padding", "3px 3px"),
                     ("border-collapse","separate"),
                     ("font-size", "16px")
                 ]
     },
     {   "selector" : "td",
         "props":[   ("color","black"),
                     ("border", "1px solid gray"),
                     ("padding", "1px 3px"),
                     ("font-size", "14px")
                 ]
     }
]

In [147]:
df

,sorgu,taban,eniyi_alis,al_sira,sat_sira,eniyi_satis,tavan,al_bek_say,al_bek_mkt,al_kum_mkt,sat_bek_say,sat_bek_mkt,sat_kum_mkt,fiyat
price,,,,,,,,,,,,,,
101.6,10:34:03.986745+00:00,101.6,116.2,112.0,,116.3,124.0,12.0,5487.0,1218276.0,,0.0,0.0,101.6
101.7,10:34:03.986745+00:00,101.6,116.2,111.0,,116.3,124.0,2.0,77.0,1212789.0,,0.0,0.0,101.7
101.8,10:34:03.986745+00:00,101.6,116.2,110.0,,116.3,124.0,2.0,87.0,1212712.0,,0.0,0.0,101.8
101.9,10:34:03.986745+00:00,101.6,116.2,109.0,,116.3,124.0,2.0,86.0,1212625.0,,0.0,0.0,101.9
102.0,10:34:03.986745+00:00,101.6,116.2,108.0,,116.3,124.0,19.0,189830.0,1212539.0,,0.0,0.0,102.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123.6,10:34:03.986745+00:00,101.6,116.2,,74.0,116.3,124.0,,0.0,0.0,2.0,89.0,252960.0,123.6
123.7,10:34:03.986745+00:00,101.6,116.2,,75.0,116.3,124.0,,0.0,0.0,3.0,48.0,253008.0,123.7
123.8,10:34:03.986745+00:00,101.6,116.2,,76.0,116.3,124.0,,0.0,0.0,3.0,113.0,253121.0,123.8


In [174]:
import pandas as pd
from datetime import datetime, timedelta

df = df_raw
df['fiyat'] = df.price
df['price'] = df['price'].apply(str)
df = df.set_index('price')


def highlight_snap(row):
    # subset=['SLA Domain Name', 'Last Snapshot']
    format_code = ''
    if row['tavan'] == row['fiyat']:
        format_code = 'background-color: rgb(25, 118, 210); font-weight: bold'
    elif row['taban'] == row['fiyat']:
        format_code = 'background-color: rgb(229, 57, 53); font-weight: bold'
    if row['eniyi_alis'] == row['fiyat']:
        format_code = 'background-color: rgb(144, 202, 249); font-weight: bold'
    elif row['eniyi_satis'] == row['fiyat']:
        format_code = 'background-color: rgb(255, 171, 145); font-weight: bold'
    #return pd.Series({'price':format_code})
    return pd.Series(format_code, row.index)

df_styler = df.style.set_table_styles(table_styler)  \
    .format_index(str.title, axis=1) 


fyt_cols = 'taban eniyi_alis fiyat eniyi_satis tavan'.split()
fmt = {x: "{:.2f}" for x in fyt_cols}
df_styler = df_styler.format(formatter=fmt, precision=0, thousands=".", decimal=",")  



df_styler = df_styler.apply(highlight_snap, axis=1, 
                            #subset=['tavan', 'taban', 'eniyi_alis', 'eniyi_satis', 'al_sira','sat_sira', 'fiyat']
                           )

df_styler          #.to_html()

,Sorgu,Taban,Eniyi_Alis,Al_Sira,Sat_Sira,Eniyi_Satis,Tavan,Al_Bek_Say,Al_Bek_Mkt,Al_Kum_Mkt,Sat_Bek_Say,Sat_Bek_Mkt,Sat_Kum_Mkt,Fiyat
price,,,,,,,,,,,,,,
101.6,11:21:26.250923+00:00,"101,60","116,20",112,,"116,30","124,00",12,5.487,1.218.276,,0,0,"101,60"
101.7,11:21:26.250923+00:00,"101,60","116,20",111,,"116,30","124,00",2,77,1.212.789,,0,0,"101,70"
101.8,11:21:26.250923+00:00,"101,60","116,20",110,,"116,30","124,00",2,87,1.212.712,,0,0,"101,80"
101.9,11:21:26.250923+00:00,"101,60","116,20",109,,"116,30","124,00",2,86,1.212.625,,0,0,"101,90"
102.0,11:21:26.250923+00:00,"101,60","116,20",108,,"116,30","124,00",19,189.830,1.212.539,,0,0,"102,00"
102.1,11:21:26.250923+00:00,"101,60","116,20",107,,"116,30","124,00",6,3.846,1.022.709,,0,0,"102,10"
102.3,11:21:26.250923+00:00,"101,60","116,20",106,,"116,30","124,00",12,31,1.018.863,,0,0,"102,30"
102.5,11:21:26.250923+00:00,"101,60","116,20",105,,"116,30","124,00",2,2.600,1.018.832,,0,0,"102,50"
102.8,11:21:26.250923+00:00,"101,60","116,20",104,,"116,30","124,00",2,625,1.016.232,,0,0,"102,80"


In [175]:
df_styler.background_gradient() \
    .bar(subset=['sat_bek_mkt', 'sat_kum_mkt'], color='#d65f5f') \
    .bar(subset=['al_bek_mkt', 'al_kum_mkt'], color='teal') 

,Sorgu,Taban,Eniyi_Alis,Al_Sira,Sat_Sira,Eniyi_Satis,Tavan,Al_Bek_Say,Al_Bek_Mkt,Al_Kum_Mkt,Sat_Bek_Say,Sat_Bek_Mkt,Sat_Kum_Mkt,Fiyat
price,,,,,,,,,,,,,,
101.6,11:21:26.250923+00:00,"101,60","116,20",112,,"116,30","124,00",12,5.487,1.218.276,,0,0,"101,60"
101.7,11:21:26.250923+00:00,"101,60","116,20",111,,"116,30","124,00",2,77,1.212.789,,0,0,"101,70"
101.8,11:21:26.250923+00:00,"101,60","116,20",110,,"116,30","124,00",2,87,1.212.712,,0,0,"101,80"
101.9,11:21:26.250923+00:00,"101,60","116,20",109,,"116,30","124,00",2,86,1.212.625,,0,0,"101,90"
102.0,11:21:26.250923+00:00,"101,60","116,20",108,,"116,30","124,00",19,189.830,1.212.539,,0,0,"102,00"
102.1,11:21:26.250923+00:00,"101,60","116,20",107,,"116,30","124,00",6,3.846,1.022.709,,0,0,"102,10"
102.3,11:21:26.250923+00:00,"101,60","116,20",106,,"116,30","124,00",12,31,1.018.863,,0,0,"102,30"
102.5,11:21:26.250923+00:00,"101,60","116,20",105,,"116,30","124,00",2,2.600,1.018.832,,0,0,"102,50"
102.8,11:21:26.250923+00:00,"101,60","116,20",104,,"116,30","124,00",2,625,1.016.232,,0,0,"102,80"


In [3]:
df_styler

,DB,Policy,Snapshot Time
0,A-DB,PROD_BACKUP,2022-10-18 12:00:00
1,B-DB,PROD_BACKUP,2022-10-16 10:00:00
2,C-DB,NONPROD_BACKUP,2022-10-15 16:00:00


In [181]:
df.dtypes

sorgu           object
taban          float64
eniyi_alis     float64
al_sira         object
sat_sira        object
eniyi_satis    float64
tavan          float64
al_bek_say      object
al_bek_mkt     float64
al_kum_mkt     float64
sat_bek_say     object
sat_bek_mkt    float64
sat_kum_mkt    float64
fiyat          float64
dtype: object